# Phase 3.1 — Local MLflow Experiment Tracking

## 1. Install MLflow



In [ ]:
%pip install "mlflow>=2.13,<4" scikit-learn pandas numpy matplotlib

## 2. Set the project path and inspect artifacts


In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path(r"C:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}\n"
        "Edit PROJECT_ROOT to the folder that contains app.py, notebooks/, tests/, and requirements.txt."
    )

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

def list_files(folder, patterns):
    folder = PROJECT_ROOT / folder
    found = []
    if folder.exists():
        for pattern in patterns:
            found.extend(folder.rglob(pattern))
    return sorted(set(p for p in found if p.is_file()))

model_files = list_files("models", ["*.pkl", "*.joblib"]) + list_files(".", ["*model*.pkl", "*model*.joblib"])
data_files = list_files("data/processed", ["*.csv", "*.parquet", "*.pkl"])
report_files = list_files("reports", ["*.csv", "*.json", "*.png", "*.jpg", "*.txt", "*.md"]) + list_files("notebooks/outputs", ["*.csv", "*.json", "*.png", "*.jpg", "*.txt", "*.md"])

print("\\nPotential model files:")
for p in sorted(set(model_files)):
    print(" -", p.relative_to(PROJECT_ROOT))
print("\\nPotential processed data files:")
for p in data_files[:80]:
    print(" -", p.relative_to(PROJECT_ROOT))
if len(data_files) > 80:
    print(f" ... and {len(data_files)-80} more")
print("\\nPotential reports / outputs:")
for p in sorted(set(report_files))[:80]:
    print(" -", p.relative_to(PROJECT_ROOT))
if len(set(report_files)) > 80:
    print(f" ... and {len(set(report_files))-80} more")

## 3. Start a local MLflow tracking server



In [ ]:
import mlflow
from pathlib import Path

MLFLOW_DIR = PROJECT_ROOT / "mlruns"
MLFLOW_DIR.mkdir(exist_ok=True)

# Use a local file-backed tracking store.
mlflow.set_tracking_uri(MLFLOW_DIR.resolve().as_uri())
EXPERIMENT_NAME = "fraud-detection-local"
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

## 4. Log a tracking smoke-test run



In [ ]:
import json
import tempfile
from pathlib import Path

with mlflow.start_run(run_name="tracking-smoke-test") as run:
    mlflow.log_param("purpose", "verify_local_mlflow_tracking")
    mlflow.log_param("dataset", "placeholder_not_evaluated")
    mlflow.log_metric("smoke_test_passed", 1.0)

    artifact = PROJECT_ROOT / "mlruns_smoke_test.json"
    artifact.write_text(json.dumps({
        "note": "This is a tracking smoke test only.",
        "not_model_evaluation": True,
        "run_id": run.info.run_id
    }, indent=2), encoding="utf-8")
    mlflow.log_artifact(str(artifact))
    artifact.unlink(missing_ok=True)

print("Smoke-test run logged successfully.")
print("Run ID:", run.info.run_id)
print("This run has placeholder metrics only; do not compare it with real model runs.")

## 5. Open the MLflow UI